Run: docker compose up -d

In [59]:
import os

from sentence_transformers import SentenceTransformer
from annoy import AnnoyIndex

In [60]:
METRIC_TYPE = "angular" # euclidean, manhattan - Która metoda ma być użyta do porównywania
# METRIC_TYPE = "euclidean"
# METRIC_TYPE = "manhattan"
# METRIC_TYPE = "dot"
TREES_NUMBER = 3       # Ilość drzew do utworzenia
TEXT_TO_SEARCH = 30     # Numer sentencji do porównania
NN_NUMBER = 5           # Ilość sąsiadów do odszukania

In [61]:
# Zestaw danych - 10 przykładowych zdań
# sentences = [
#     "Słońce świeci na niebie.", # 0
#     "Dzisiaj jest piękna pogoda.",
#     "Lubię uczyć się programowania.",
#     "Python to popularny język programowania.",
#     "Wieczorem często czytam książki.",
#     "Mój pies lubi biegać po parku.",
#     "Programowanie to świetna zabawa.",
#     "Niebo jest dziś bardzo niebieskie.",
#     "Czytanie rozwija wyobraźnię.",
#     "Chodzę codziennie na spacer z psem.",
#     "Lubię Mazury, morze i góry.", # 10
#     "Java to mój ulubiony język programowania.",
#     "Ostatnio częściej programuję w Pythonie",
#     "SQL też nie jest zły.",
#     "Za tydzień jedziemy całą ekipą na wakacje w góry.",
#     "Zawsze na wyjazdy wakacyjne zabieramy naszego psa.",
#     "Dostałem wczoraj mandat za przektoczenie predkości.",
#     "Jazda moim Porshe wyzwala we mnie mega emocje.",
#     "Kolarstwo górskie daje mi wiele frajdy.",
#     "Wczoraj byłem w kinie.",
#     "Mam uczulenie na czekoladę ale za to mogę jeść marmoladę.", # 20
#     "Najlepsze pączki są z marmoladą i lukrem.",
#     "Twój tort urodzinowy był pyszny.",
#     "Od roku ograniczam cukier i czuję sie wyśmienicie.",
#     "Wczoraj kupiłem poradnik pt. Dieta dla każdego.",
#     "Sposób odżywiania ma wpływ na nasze samopoczucie.",
#     "Właśnie skończyłem czytać fajną książkę.",
#     "Czytanie książek to moja ulubiona zabawa.",
#     "Wczoraj byłem w bibliotek i wypożyczyłem kilka książek na temat programowania.",
#     "Wczoraj wróciliśmy z wakacji z Mazur.",
#     "Mój pies lubi gonić koty." # 30
# ]

sentences = [
    "Wireless noise cancelling headphones with long battery life",
    "Bluetooth over-ear headphones with active noise cancellation",
    "Portable wireless speaker with deep bass and Bluetooth connectivity",
    "Smartphone with high resolution camera and fast processor",
    "Gaming laptop with powerful GPU and high refresh rate display",
    "Lightweight laptop for everyday work and browsing",
    "Mechanical keyboard with RGB lighting for gaming",
    "Ergonomic wireless mouse with adjustable DPI",
    "4K monitor with ultra-thin bezels and HDR support",
    "USB-C docking station for multiple devices",

    "Running shoes designed for long distance comfort",
    "Lightweight sneakers for everyday casual wear",
    "Breathable sports t-shirt for training sessions",
    "Fitness tracker with heart rate monitoring",
    "Smartwatch with activity tracking and notifications",
    "Yoga mat with non-slip surface",
    "Dumbbells set for strength training at home",
    "Resistance bands for full body workout",
    "Cycling helmet with aerodynamic design",
    "Sports water bottle with leak-proof design",

    "Modern coffee machine with programmable settings",
    "Automatic espresso machine with milk frother",
    "Compact vacuum cleaner with strong suction power",
    "Robot vacuum cleaner with smart navigation",
    "Air purifier with HEPA filter for home use",
    "Table lamp with adjustable brightness levels",
    "LED desk lamp with touch control",
    "Smart thermostat for energy saving",
    "Electric kettle with temperature control",
    "Blender for smoothies and healthy drinks",

    "Casual denim jeans with comfortable fit",
    "Stylish leather jacket for autumn season",
    "Winter jacket with thermal insulation",
    "Classic white t-shirt made from cotton",
    "Sneakers with modern design and comfort",
    "Elegant dress for special occasions",
    "Hoodie with soft fabric and relaxed fit",
    "Running shorts for summer workouts",
    "Formal shirt for business meetings",
    "Sports jacket with breathable material",

    "Bestselling novel with engaging storyline",
    "Science fiction book with futuristic themes",
    "Cookbook with healthy recipes and tips",
    "Biography of a famous entrepreneur",
    "Self-help book about productivity and habits",
    "Guide to learning programming for beginners",
    "Fantasy novel with magical world",
    "Thriller book with unexpected twists",
    "Educational book for children",
    "Travel guide for exploring Europe",

    "Educational toy for kids with interactive features",
    "Building blocks set for creative play",
    "Puzzle game for brain training",
    "Board game for family entertainment",
    "Remote control car with high speed",
    "Interactive robot toy with sensors",
    "Plush toy for children",
    "STEM learning kit for kids",
    "Toy train set with tracks",
    "Outdoor play set for children",

    "Noise cancelling earbuds with compact design",
    "Wireless earbuds with long battery life",
    "Portable charger with fast charging support",
    "Smartphone case with shock protection",
    "Tablet device for media consumption",
    "Gaming console with high performance graphics",
    "VR headset for immersive gaming experience",
    "Smart home hub with voice control",
    "Wireless router with high speed internet",
    "External SSD with fast data transfer",

    "Trail running shoes for outdoor adventures",
    "Hiking backpack with large capacity",
    "Camping tent for 2 people",
    "Sleeping bag for cold weather",
    "Portable gas stove for camping",
    "Climbing shoes with strong grip",
    "Fitness gloves for weight lifting",
    "Jump rope for cardio training",
    "Exercise bike for home workouts",
    "Treadmill with adjustable speed",

    "Smart LED light bulbs with app control",
    "Home security camera with motion detection",
    "Door lock with fingerprint access",
    "Video doorbell with HD camera",
    "Smart plug with remote control",
    "Air fryer for healthy cooking",
    "Microwave oven with multiple functions",
    "Dishwasher with energy saving mode",
    "Refrigerator with large storage capacity",
    "Washing machine with quick wash program"
]

In [62]:
# Ładowanie modelu do tworzenia osadzeń (embeddingów) zdań
model = SentenceTransformer('all-MiniLM-L6-v2')  # lekki i szybki model

# Tworzenie embeddingów dla każdego zdania
embeddings = model.encode(sentences)  # embedding = wektor liczb reprezentujących znaczenie zdania
vector_dim = len(embeddings[0])       # wymiar embeddingu, np. 384

# Ścieżka do pliku z indeksem
index_file = "index_annoy/sentences.ann"

# Tworzenie i budowanie indeksu tylko jeśli nie istnieje
if not os.path.exists(index_file):
    index = AnnoyIndex(vector_dim, METRIC_TYPE)
    for i, vec in enumerate(embeddings):
        index.add_item(i, vec)
    # Budowanie drzewa wyszukiwania (10 drzew – więcej = lepsza jakość, wolniejsze budowanie
    index.build(TREES_NUMBER)
    index.save(index_file)
    print(f"Indeks zapisany do pliku: {index_file}")
else:
    print(f"Indeks już istnieje: {index_file}")

# Tworzenie indeksu Annoy z metryką 'angular' (opartą na kącie między wektorami)
index = AnnoyIndex(vector_dim, METRIC_TYPE)
index.load(index_file)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indeks zapisany do pliku: index_annoy/sentences.ann


True

In [63]:
# Przykład zapytania – wyszukujemy zdania podobne do zdania nr 2
query_id = TEXT_TO_SEARCH
print(f"Zapytanie: {sentences[query_id]}")

Zapytanie: Casual denim jeans with comfortable fit


In [64]:
# Wyszukiwanie 3 najbardziej podobnych zdań (bez samego zapytania)
nearest_ids = index.get_nns_by_item(query_id, NN_NUMBER, include_distances=True)

# Wyświetlenie wyników
print(f"{METRIC_TYPE} - Najbardziej podobne zdania:")
for idx, dist in zip(*nearest_ids):
    print(f"- {sentences[idx]} - (odległość: {dist:.4f})")


angular - Najbardziej podobne zdania:
- Casual denim jeans with comfortable fit - (odległość: 0.0000)
- Hoodie with soft fabric and relaxed fit - (odległość: 0.9294)
- Lightweight sneakers for everyday casual wear - (odległość: 1.0204)
- Sneakers with modern design and comfort - (odległość: 1.0835)
- Stylish leather jacket for autumn season - (odległość: 1.1258)
